## Avant de collecter : les 6 questions

| Question | Ma réponse |
|---|---|
| Une API ou un open data existent-ils ? | Non, le site est fait exprès pour apprendre le scraping (« We love being scraped! ») |
| Que dit le robots.txt ? | Il est vide : rien n'est interdit, pas de Crawl-delay |
| Que disent les CGU ? | Pas de conditions d'utilisation, site d'entraînement |
| Y a-t-il des données personnelles ? | Non : titres, prix, stocks, notes. Aucun nom de personne |
| Combien de requêtes, à quel rythme ? | 1 page de liste + 19 fiches = **20 requêtes**, avec 1 seconde de pause, soit environ 20 secondes |
| Est-ce que je garde le résultat ? | Oui, dans `data/poetry.csv`, pour ne pas refaire la collecte |

# Collecte de la catégorie Poetry sur books.toscrape.com

## Ma fiche technique

- Adresse de départ : https://books.toscrape.com/catalogue/category/books/poetry_23/index.html
- Nombre de livres annoncé par le site : **19**
- Nombre de pages : **1** (pas de « Page 1 of N » ni de bouton « next »)

| Donnée | Sélecteur | Contenu ou attribut |
|---|---|---|
| Bloc qui se répète (1 livre) | `article.product_pod` | 19 blocs sur la page |
| Titre complet | `h3 a` | attribut `title` |
| Prix | `p.price_color` | contenu |
| Disponibilité | `p.instock` | contenu |
| Note | `p.star-rating` | 2e mot de l'attribut `class` (One à Five) |
| Lien vers la fiche | `h3 a` | attribut `href` (lien relatif) |
| Page suivante | `li.next a` | absent : une seule page |

Sur la fiche détaillée de chaque livre :

| Donnée | Sélecteur | Contenu ou attribut |
|---|---|---|
| UPC et stock | `table.table-striped tr` (th + td) | contenu |
| Catégorie | `ul.breadcrumb li a` (le dernier) | contenu |

In [1]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import os

print("Tout est bien installé")
print(os.getcwd())

Tout est bien installé
c:\Users\nulen\Documents\lire-collecter-une-page\notebooks


In [2]:
import time
from urllib.parse import urljoin

URL = "https://books.toscrape.com/catalogue/category/books/poetry_23/index.html"
ENTETES = {"User-Agent": "bootcamp-data/1.0 (nulendark@gmail.com)"}

r = requests.get(URL, headers=ENTETES, timeout=10)
r.encoding = "utf-8"

print("status :", r.status_code)
print("le prix du 1er livre est présent :", "£51.77" in r.text)
print("le titre du 1er livre est présent :", "A Light in the Attic" in r.text)

status : 200
le prix du 1er livre est présent : True
le titre du 1er livre est présent : True


In [3]:
soup = BeautifulSoup(r.text, "html.parser")
fiches = soup.select("article.product_pod")
print("fiches trouvées :", len(fiches))

NOTES = {"One": 1, "Two": 2, "Three": 3, "Four": 4, "Five": 5}

def lire_fiche(fiche, url_page):
    lien = fiche.select_one("h3 a")
    classes = fiche.select_one("p.star-rating")["class"]
    return {
        "titre": lien["title"],
        "prix": fiche.select_one("p.price_color").get_text(strip=True),
        "dispo": fiche.select_one("p.instock").get_text(strip=True),
        "note": NOTES[classes[1]],
        "url": urljoin(url_page, lien["href"]),
    }

livres_page1 = [lire_fiche(f, URL) for f in fiches]
df_page1 = pd.DataFrame(livres_page1)
df_page1

fiches trouvées : 19


,titre,prix,dispo,note,url
0,A Light in the Attic,£51.77,In stock,3,https://books.toscrape.com/catalogue/a-light-i...
1,The Black Maria,£52.15,In stock,1,https://books.toscrape.com/catalogue/the-black...
2,Shakespeare's Sonnets,£20.66,In stock,4,https://books.toscrape.com/catalogue/shakespea...
3,Olio,£23.88,In stock,1,https://books.toscrape.com/catalogue/olio_984/...
4,You can't bury them all: Poems,£33.63,In stock,2,https://books.toscrape.com/catalogue/you-cant-...
5,Slow States of Collapse: Poems,£57.31,In stock,3,https://books.toscrape.com/catalogue/slow-stat...
6,Untitled Collection: Sabbath Poems 2014,£14.27,In stock,4,https://books.toscrape.com/catalogue/untitled-...
7,Poems That Make Grown Women Cry,£14.19,In stock,4,https://books.toscrape.com/catalogue/poems-tha...
8,Night Sky with Exit Wounds,£41.05,In stock,1,https://books.toscrape.com/catalogue/night-sky...
9,salt.,£46.78,In stock,4,https://books.toscrape.com/catalogue/salt_731/...


In [4]:
print("titres tronqués :", df_page1["titre"].str.contains("...", regex=False).sum())

print("note la plus basse :", df_page1["note"].min())
print("note la plus haute :", df_page1["note"].max())
print("type de la note :", df_page1["note"].dtype)

print(df_page1.loc[10, "titre"])

titres tronqués : 0
note la plus basse : 1
note la plus haute : 5
type de la note : int64
Quarter Life Poetry: Poems for the Young, Broke and Hangry


In [5]:
def collecter_toutes_les_pages(url_depart, pages_max=10):
    url = url_depart
    livres = []
    page = 0

    while url and page < pages_max:
        page += 1
        r = requests.get(url, headers=ENTETES, timeout=10)
        r.encoding = "utf-8"
        if r.status_code != 200:
            print(f"  page {page} : status {r.status_code}, on s'arrête")
            break

        soup = BeautifulSoup(r.text, "html.parser")
        fiches = soup.select("article.product_pod")
        livres += [lire_fiche(f, url) for f in fiches]
        print(f"  page {page} : {len(fiches)} livres  (total {len(livres)})")

        suivant = soup.select_one("li.next a")
        url = urljoin(url, suivant["href"]) if suivant else None

        time.sleep(1)

    return livres


livres = collecter_toutes_les_pages(URL)
print("collecte :", len(livres), "livres")
print("nombre annoncé par le site : 19")
print("c'est bon :", len(livres) == 19)

  page 1 : 19 livres  (total 19)
collecte : 19 livres
nombre annoncé par le site : 19
c'est bon : True


In [6]:
import re

def lire_detail(url):
    r = requests.get(url, headers=ENTETES, timeout=10)
    r.encoding = "utf-8"
    if r.status_code != 200:
        return {"categorie": None, "stock": None, "upc": None}

    soup = BeautifulSoup(r.text, "html.parser")

    infos = {}
    for ligne in soup.select("table.table-striped tr"):
        cle = ligne.select_one("th")
        val = ligne.select_one("td")
        if cle and val:
            infos[cle.get_text(strip=True)] = val.get_text(strip=True)

    miettes = soup.select("ul.breadcrumb li a")
    categorie = miettes[-1].get_text(strip=True) if miettes else None

    return {
        "categorie": categorie,
        "stock": infos.get("Availability"),
        "upc": infos.get("UPC"),
    }


def nombre_en_stock(texte):
    if not texte:
        return None
    trouve = re.search(r"\((\d+) available\)", texte)
    return int(trouve.group(1)) if trouve else 0


for i, livre in enumerate(livres, start=1):
    detail = lire_detail(livre["url"])
    livre.update(detail)
    print(f"  {i}/{len(livres)} : {livre['titre'][:40]} -> {detail['stock']}")
    time.sleep(1)

print("fiches visitées :", len(livres))

  1/19 : A Light in the Attic -> In stock (22 available)
  2/19 : The Black Maria -> In stock (19 available)
  3/19 : Shakespeare's Sonnets -> In stock (19 available)
  4/19 : Olio -> In stock (19 available)
  5/19 : You can't bury them all: Poems -> In stock (17 available)
  6/19 : Slow States of Collapse: Poems -> In stock (17 available)
  7/19 : Untitled Collection: Sabbath Poems 2014 -> In stock (16 available)
  8/19 : Poems That Make Grown Women Cry -> In stock (15 available)
  9/19 : Night Sky with Exit Wounds -> In stock (15 available)
  10/19 : salt. -> In stock (14 available)
  11/19 : Quarter Life Poetry: Poems for the Young -> In stock (14 available)
  12/19 : Out of Print: City Lights Spotlight No.  -> In stock (8 available)
  13/19 : Les Fleurs du Mal -> In stock (8 available)
  14/19 : Howl and Other Poems -> In stock (8 available)
  15/19 : Leave This Song Behind: Teen Poetry at I -> In stock (7 available)
  16/19 : The Collected Poems of W.B. Yeats (The C -> In stock (6

In [7]:
df = pd.DataFrame(livres)

df["prix_gbp"] = df["prix"].str.replace("£", "").astype(float)

df["stock"] = df["stock"].apply(nombre_en_stock)

df = df[["titre", "categorie", "prix_gbp", "note", "stock", "upc", "url"]]

print("nombre de lignes :", len(df))
print("UPC différents :", df["upc"].nunique())
print("notes vides :", df["note"].isna().sum())
print()
print(df.dtypes)

df

nombre de lignes : 19
UPC différents : 19
notes vides : 0

titre            str
categorie        str
prix_gbp     float64
note           int64
stock          int64
upc              str
url              str
dtype: object


,titre,categorie,prix_gbp,note,stock,upc,url
0,A Light in the Attic,Poetry,51.77,3,22,a897fe39b1053632,https://books.toscrape.com/catalogue/a-light-i...
1,The Black Maria,Poetry,52.15,1,19,1dfe412b8ac00530,https://books.toscrape.com/catalogue/the-black...
2,Shakespeare's Sonnets,Poetry,20.66,4,19,30a7f60cd76ca58c,https://books.toscrape.com/catalogue/shakespea...
3,Olio,Poetry,23.88,1,19,feb7cc7701ecf901,https://books.toscrape.com/catalogue/olio_984/...
4,You can't bury them all: Poems,Poetry,33.63,2,17,55f9da0c5eea2e10,https://books.toscrape.com/catalogue/you-cant-...
5,Slow States of Collapse: Poems,Poetry,57.31,3,17,b4fd5943413e089a,https://books.toscrape.com/catalogue/slow-stat...
6,Untitled Collection: Sabbath Poems 2014,Poetry,14.27,4,16,657fe5ead67a7767,https://books.toscrape.com/catalogue/untitled-...
7,Poems That Make Grown Women Cry,Poetry,14.19,4,15,31001a1210623bd4,https://books.toscrape.com/catalogue/poems-tha...
8,Night Sky with Exit Wounds,Poetry,41.05,1,15,cfeb1e1e0e695b58,https://books.toscrape.com/catalogue/night-sky...
9,salt.,Poetry,46.78,4,14,86cbddb61ea78bb7,https://books.toscrape.com/catalogue/salt_731/...


In [8]:
print("nombre de livres :", len(df))
print("stock total :", df["stock"].sum(), "exemplaires")
print()

print("prix moyen :", round(df["prix_gbp"].mean(), 2), "£")
plus_cher = df.loc[df["prix_gbp"].idxmax()]
print("livre le plus cher :", plus_cher["titre"], "à", plus_cher["prix_gbp"], "£")
print()

print("répartition des notes :")
print(df["note"].value_counts().sort_index())
print()

print("catégories différentes :", df["categorie"].nunique())
print(df["categorie"].unique())

nombre de livres : 19
stock total : 235 exemplaires

prix moyen : 35.97 £
livre le plus cher : Slow States of Collapse: Poems à 57.31 £

répartition des notes :
note
1    3
2    2
3    2
4    6
5    6
Name: count, dtype: int64

catégories différentes : 1
<StringArray>
['Poetry']
Length: 1, dtype: str


## Mes réponses 

**1. Combien de livres, et quel stock total ?**
J'ai collecté 19 livres comme annoncé sur le site. Il y a 235 exemplaires en stock au total de 22 pour A Light in the Attic à 1 seul pour Twenty Love Poems and a Song of Despair.

**2. Quel est le prix moyen, et le livre le plus cher ?**
Le prix moyen est de 35,97 £. Le livre le plus cher est Slow States of Collapse: Poems à 57,31 £.

**3. Comment se répartissent les notes ?**

| Note | Nombre de livres |
|---|---|
| 1 | 3 |
| 2 | 2 |
| 3 | 2 |
| 4 | 6 |
| 5 | 6 |

Les livres de poésie sont plutôt bien notés ici 12 livres sur 19 ont 4 ou 5 étoiles. Seulement 3 livres ont la note la plus basse.

**4. La colonne categorie contient-elle une seule valeur ?**
Oui une seule : Poetry. Ça confirme que ma collecte n'a récupéré que des livres de ma catégorie. Si j'avais trouvé plusieurs valeurs, ça aurait voulu dire que mon script avait suivi des liens vers d'autres catégories, ou que mon sélecteur du fil d'Ariane prenait le mauvais élément.

In [9]:
df.to_csv("../data/poetry.csv", index=False)

verif = pd.read_csv("../data/poetry.csv")
print("lignes dans le CSV :", len(verif))
print("colonnes :", list(verif.columns))

lignes dans le CSV : 19
colonnes : ['titre', 'categorie', 'prix_gbp', 'note', 'stock', 'upc', 'url']
